# Advanced Pitch Statistics from Statcast

This notebook explores the advanced pitch-level data used to build pitcher and batter profiles.

**Pitcher Arsenal Stats:**
- Velocity, spin, movement by pitch type
- Pitch usage percentages
- Whiff%, CSW%, zone%, chase induced

**Batter Profile Stats:**
- Performance vs pitch types (fastball, breaking, offspeed)
- Performance vs velocity buckets (90-93, 94-96, 97+)
- Performance vs movement profiles (high rise, heavy sweep, heavy drop)
- Contact quality (exit velo, xwOBA)

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_pitcher_pitch_type_stats,
    get_batter_pitch_stats,
    get_batter_pitch_type_stats,
    get_plate_appearances,
)

# Import config
from src.config import (
    DATA_START,
    DATA_END,
    DATA_DIR,
)

pd.set_option('display.max_columns', 50)

print(f"Date range: {DATA_START} to {DATA_END}")

Date range: 2021-04-01 to 2026-03-31


## 1. Raw Statcast Data

First, let's fetch the raw pitch-level data and explore what's available.

In [2]:
from pathlib import Path

# Load pitches from notebook 01 output
pitches_path = Path(f"../{DATA_DIR}/raw/pitches.parquet")

if pitches_path.exists():
    pitches = pd.read_parquet(pitches_path)
    print(f"Loaded pitches from {pitches_path}")
else:
    print(f"Pitches file not found at {pitches_path}")
    print("Run notebook 01_compile_data first to fetch the data.")
    raise FileNotFoundError(f"Run notebook 01 first: {pitches_path}")

print(f"Shape: {pitches.shape}")
print(f"Date range: {pitches['game_date'].min()} to {pitches['game_date'].max()}")
pitches.head()

Loaded pitches from ../data/raw/pitches.parquet
Shape: (3860179, 118)
Date range: 2021-04-01 00:00:00 to 2026-03-30 00:00:00


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,spin_dir,spin_rate_deprecated,break_angle_deprecated,break_length_deprecated,zone,des,game_type,stand,p_throws,home_team,away_team,type,hit_location,bb_type,balls,...,delta_pitcher_run_exp,hyper_speed,home_score_diff,bat_score_diff,home_win_exp,bat_win_exp,age_pit_legacy,age_bat_legacy,age_pit,age_bat,n_thruorder_pitcher,n_priorpa_thisgame_player_at_bat,pitcher_days_since_prev_game,batter_days_since_prev_game,pitcher_days_until_next_game,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,SL,2021-04-14,83.6,-1.6,6.17,"Jansen, Kenley",606132,445276,strikeout,swinging_strike_blocked,<NA>,<NA>,<NA>,<NA>,13,"Raimel Tapia strikes out swinging, catcher Aus...",R,L,R,LAD,COL,S,2,None,0,...,0.168,<NA>,2,-2,0.975,0.025,33,27,34,27,1,4,3,1,2,1,3.43,-0.85,0.85,67.7,<NA>,<NA>,<NA>,<NA>,<NA>
1,FC,2021-04-14,93.6,-1.73,6.27,"Jansen, Kenley",606132,445276,None,foul,<NA>,<NA>,<NA>,<NA>,11,Foul,R,L,R,LAD,COL,S,<NA>,None,0,...,0.0,88.0,2,-2,0.975,0.025,33,27,34,27,1,4,3,1,2,1,0.99,-0.48,0.48,66.9,<NA>,<NA>,<NA>,<NA>,<NA>
2,SI,2021-04-14,94.0,-1.52,6.19,"Jansen, Kenley",606132,445276,None,foul,<NA>,<NA>,<NA>,<NA>,1,Foul,R,L,R,LAD,COL,S,<NA>,None,0,...,0.077,94.1,2,-2,0.967,0.033,33,27,34,27,1,4,3,1,2,1,1.01,0.48,-0.48,67.7,<NA>,<NA>,<NA>,<NA>,<NA>
3,FC,2021-04-14,93.6,-1.6,6.22,"Jansen, Kenley",606132,445276,None,called_strike,<NA>,<NA>,<NA>,<NA>,5,Called Strike,R,L,R,LAD,COL,S,<NA>,None,0,...,0.049,<NA>,2,-2,0.964,0.036,33,27,34,27,1,4,3,1,2,1,1.12,-0.62,0.62,67.6,<NA>,<NA>,<NA>,<NA>,<NA>
4,FC,2021-04-14,89.8,-1.7,6.19,"Jansen, Kenley",641658,445276,strikeout,swinging_strike,<NA>,<NA>,<NA>,<NA>,3,Garrett Hampson strikes out swinging.,R,R,R,LAD,COL,S,2,None,3,...,0.324,<NA>,2,-2,0.897,0.103,33,26,34,27,1,4,3,1,2,1,1.48,-0.78,-0.78,68.6,<NA>,<NA>,<NA>,<NA>,<NA>


In [3]:
# Key pitch characteristic columns
pitch_cols = [
    'pitch_type', 'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'release_extension',
    'plate_x', 'plate_z', 'zone',
    'description', 'events', 'type',
]

pitches[pitch_cols].head(20)

,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,release_extension,plate_x,plate_z,zone,description,events,type
0,SL,83.6,2577,0.85,-0.2,7.1,-0.714857,0.618207,13,swinging_strike_blocked,strikeout,S
1,FC,93.6,2753,0.48,1.62,7.1,-0.874797,2.697399,11,foul,None,S
2,SI,94.0,2415,-0.48,1.55,7.3,-0.766592,2.990841,1,foul,None,S
3,FC,93.6,2665,0.62,1.48,7.5,0.187778,2.668229,5,called_strike,None,S
4,FC,89.8,2620,0.78,1.33,7.5,0.41017,3.055393,3,swinging_strike,strikeout,S
5,FC,89.5,2688,0.54,1.71,7.5,0.425914,2.794708,3,foul,None,S
6,FC,90.0,2673,0.39,1.8,7.5,0.82191,2.753804,6,swinging_strike,None,S
7,SI,91.2,2411,-0.2,1.96,7.6,-0.045895,4.584301,11,ball,None,B
8,SI,93.1,2371,-0.38,1.97,7.6,1.104804,2.140797,14,ball,None,B
9,FC,91.5,2819,0.54,1.67,7.3,0.924303,2.081029,14,called_strike,None,S


In [4]:
# Pitch type distribution
print("Pitch type distribution:")
pitch_counts = pitches['pitch_type'].value_counts()
pitch_pcts = pitches['pitch_type'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': pitch_counts,
    'pct': pitch_pcts.round(1)
}).head(15)

Pitch type distribution:


,count,pct
pitch_type,,
FF,1243199,32.9
SL,594819,15.8
SI,586387,15.5
CH,405137,10.7
FC,290976,7.7
CU,258885,6.9
ST,201908,5.3
FS,89339,2.4
KC,76964,2.0


In [5]:
# Pitch outcome descriptions
print("Pitch descriptions (outcomes):")
pitches['description'].value_counts()

Pitch descriptions (outcomes):


description
ball                       1281221
foul                        681645
hit_into_play               677357
called_strike               628136
swinging_strike             414931
blocked_ball                 84001
foul_tip                     38029
swinging_strike_blocked      22979
automatic_ball               11891
hit_by_pitch                 11294
foul_bunt                     6465
missed_bunt                   1232
automatic_strike               624
pitchout                       236
bunt_foul_tip                  135
foul_pitchout                    2
intent_ball                      1
Name: count, dtype: int64

## 2. Pitcher Arsenal Profiles

Aggregate pitch-level data into pitcher profiles with:
- Fastball characteristics (velocity, spin, movement)
- Breaking ball characteristics
- Offspeed characteristics
- Overall effectiveness metrics (whiff%, CSW%, etc.)

In [6]:
# Get pitcher arsenal stats (uses cached pitches)
pitcher_arsenal = get_pitcher_arsenal(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitchers with arsenal data: {len(pitcher_arsenal)}")
pitcher_arsenal.head(10)

Computing pitcher arsenal stats...
  Computed arsenal for 1618 pitchers
Pitchers with arsenal data: 1618


,pitcher_id,pitcher_name,p_throws,total_pitches,primary_pitch,ff_usage_pct,si_usage_pct,si_velo_avg,si_spin_avg,si_vmov_avg,si_hmov_avg,si_whiff_pct,fc_usage_pct,sl_usage_pct,sl_velo_avg,sl_spin_avg,sl_vmov_avg,sl_hmov_avg,sl_whiff_pct,cu_usage_pct,ch_usage_pct,sv_usage_pct,fs_usage_pct,kc_usage_pct,st_usage_pct,...,st_velo_avg,st_spin_avg,st_vmov_avg,st_hmov_avg,st_whiff_pct,fs_velo_avg,fs_spin_avg,fs_vmov_avg,fs_hmov_avg,fs_whiff_pct,kc_velo_avg,kc_spin_avg,kc_vmov_avg,kc_hmov_avg,kc_whiff_pct,sv_velo_avg,sv_spin_avg,sv_vmov_avg,sv_hmov_avg,sv_whiff_pct,kn_velo_avg,kn_spin_avg,kn_vmov_avg,kn_hmov_avg,kn_whiff_pct
0,424144,"Pérez, Oliver",L,144,SI,0.118056,0.569444,88.632927,2107.890244,0.643780,1.493171,0.113636,0.000000,0.312500,76.324444,2197.933333,-0.161778,-0.772889,0.210526,0.000000,0.000000,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,425794,"Wainwright, Adam",R,8175,CU,0.095413,0.288440,88.346735,2202.328879,0.995619,-1.079330,0.075196,0.231070,0.000979,NaN,NaN,NaN,NaN,NaN,0.317554,0.060550,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,425844,"Greinke, Zack",R,7285,FF,0.353878,0.076733,89.694097,2192.044723,0.839374,-1.168676,0.069869,0.069321,0.143171,81.386481,2421.378482,0.268658,0.870566,0.251429,0.167193,0.187234,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,429722,"Santana, Ervin",R,1063,SL,0.451552,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.515522,84.362044,2319.118613,0.151168,0.365858,0.339623,0.000000,0.032926,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,431148,"Kazmir, Scott",L,207,FF,0.541063,0.004831,NaN,NaN,NaN,NaN,NaN,0.000000,0.144928,87.143333,1946.034483,0.675000,0.098667,0.277778,0.000000,0.309179,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,433589,"Petit, Yusmeiro",R,1082,FF,0.458410,0.000000,NaN,NaN,NaN,NaN,NaN,0.228281,0.000000,NaN,NaN,NaN,NaN,NaN,0.135860,0.176525,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,434378,"Verlander, Justin",R,10542,FF,0.488902,0.001423,NaN,NaN,NaN,NaN,NaN,0.000000,0.247676,87.119763,2480.340882,0.403305,0.347254,0.294989,0.181749,0.058243,0.0,0.0,0.0,0.022007,...,80.510345,2655.702586,-0.144224,0.967284,0.238095,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,434671,"Sánchez, Aníbal",R,1232,FC,0.174513,0.184253,89.120705,1921.017621,1.108414,-1.036211,0.067308,0.272727,0.056818,83.018571,2179.142857,0.363571,0.427286,0.400000,0.056006,0.255682,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,445213,"Kintzler, Brandon",R,510,SI,0.005882,0.745098,92.044737,1968.957895,0.887079,-1.334895,0.115152,0.000000,0.139216,85.359155,2350.211268,0.296056,0.080423,0.343750,0.000000,0.109804,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,445276,"Jansen, Kenley",R,4699,FC,0.000000,0.162375,93.794364,2295.821288,1.588873,-0.536540,0.286096,0.717812,0.110023,82.390522,2444.988350,-0.434139,0.500000,0.293173,0.000000,0.000000,0.0,0.0,0.0,0.009789,...,81.571739,2533.739130,0.179783,1.086957,0.160000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# All available arsenal columns
print("Pitcher arsenal columns:")
print(pitcher_arsenal.columns.tolist())

Pitcher arsenal columns:
['pitcher_id', 'pitcher_name', 'p_throws', 'total_pitches', 'primary_pitch', 'ff_usage_pct', 'si_usage_pct', 'si_velo_avg', 'si_spin_avg', 'si_vmov_avg', 'si_hmov_avg', 'si_whiff_pct', 'fc_usage_pct', 'sl_usage_pct', 'sl_velo_avg', 'sl_spin_avg', 'sl_vmov_avg', 'sl_hmov_avg', 'sl_whiff_pct', 'cu_usage_pct', 'ch_usage_pct', 'sv_usage_pct', 'fs_usage_pct', 'kc_usage_pct', 'st_usage_pct', 'kn_usage_pct', 'whiff_pct', 'swstr_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced', 'contact_pct', 'f_strike_pct', 'ff_velo_avg', 'ff_spin_avg', 'ff_vmov_avg', 'ff_hmov_avg', 'ff_whiff_pct', 'fc_velo_avg', 'fc_spin_avg', 'fc_vmov_avg', 'fc_hmov_avg', 'fc_whiff_pct', 'cu_velo_avg', 'cu_spin_avg', 'cu_vmov_avg', 'cu_hmov_avg', 'cu_whiff_pct', 'ch_velo_avg', 'ch_spin_avg', 'ch_vmov_avg', 'ch_hmov_avg', 'ch_whiff_pct', 'st_velo_avg', 'st_spin_avg', 'st_vmov_avg', 'st_hmov_avg', 'st_whiff_pct', 'fs_velo_avg', 'fs_spin_avg', 'fs_vmov_avg', 'fs_hmov_avg', 'fs_whiff_pct', 'kc_velo_avg'

In [8]:
# Summary statistics for key arsenal metrics
arsenal_stats = [
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'fb_vmov_avg',
    'brk_velo_avg', 'brk_hmov_avg', 'brk_vmov_avg',
    'off_velo_avg', 'off_velo_diff',
    'fb_usage_pct', 'brk_usage_pct', 'off_usage_pct',
    'whiff_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced',
]

available_stats = [c for c in arsenal_stats if c in pitcher_arsenal.columns]
pitcher_arsenal[available_stats].describe().round(2)

,whiff_pct,csw_pct,zone_pct,chase_pct_induced
count,1618.00,1618.00,1618.00,1618.00
mean,0.23,0.27,0.48,0.27
std,0.06,0.04,0.04,0.05
min,0.00,0.06,0.17,0.03
25%,0.19,0.25,0.46,0.24
50%,0.23,0.27,0.49,0.27
75%,0.26,0.29,0.51,0.30
max,0.53,0.38,0.68,0.45


## 3. Pitcher Stats by Pitch Type

Detailed breakdown for each pitch type a pitcher throws.

In [9]:
# Get per-pitch-type stats for pitchers
pitcher_pitch_types = get_pitcher_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=20,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Pitcher-pitch type combinations: {len(pitcher_pitch_types)}")
pitcher_pitch_types.head(15)

Computing pitcher pitch-type stats...
  Computed stats for 6632 pitcher-pitch combinations
Pitcher-pitch type combinations: 6632


,pitcher_id,pitcher_name,p_throws,pitch_type,pitch_count,usage_pct,velo_avg,velo_max,spin_avg,vmov_avg,hmov_avg,extension,whiff_pct,csw_pct,swing_pct,avg_exit_velo,avg_launch_angle,xwoba_against,xba_against
0,424144,"Pérez, Oliver",L,SI,82,0.569444,88.632927,91.9,2107.890244,0.643780,1.493171,6.219512,0.113636,0.231707,0.536585,88.014815,9.407407,0.379178,0.311642
1,424144,"Pérez, Oliver",L,SL,45,0.312500,76.324444,79.1,2197.933333,-0.161778,-0.772889,6.031111,0.210526,0.377778,0.422222,NaN,NaN,NaN,NaN
2,425794,"Wainwright, Adam",R,CH,495,0.060550,82.281414,85.7,1722.395918,0.668141,-1.169253,6.468024,0.160804,0.202020,0.402020,86.911765,4.901961,0.308929,0.280569
3,425794,"Wainwright, Adam",R,CS,49,0.005994,67.059184,70.4,2556.285714,-1.232449,1.296735,6.14898,0.272727,0.326531,0.224490,NaN,NaN,NaN,NaN
4,425794,"Wainwright, Adam",R,CU,2596,0.317554,72.850462,77.1,2774.716127,-1.211102,1.363532,6.322616,0.219780,0.318567,0.455701,85.488555,10.195122,0.363505,0.328343
5,425794,"Wainwright, Adam",R,FC,1889,0.231070,84.212176,89.8,2390.914654,0.546236,0.514002,6.469796,0.173333,0.246162,0.516146,86.548762,11.445545,0.369198,0.324933
6,425794,"Wainwright, Adam",R,FF,780,0.095413,87.859103,92.3,2216.473272,1.233038,-0.131449,6.537273,0.120879,0.253846,0.350000,90.236434,11.658915,0.414309,0.364015
7,425794,"Wainwright, Adam",R,SI,2358,0.288440,88.346735,92.8,2202.328879,0.995619,-1.079330,6.538518,0.075196,0.282867,0.377863,91.784807,11.705215,0.410986,0.353190
8,425844,"Greinke, Zack",R,CH,1364,0.187234,86.464516,90.2,1657.978629,0.345579,-1.074655,5.946188,0.214391,0.147361,0.499267,87.129487,-5.599359,0.299723,0.290524
9,425844,"Greinke, Zack",R,CU,1218,0.167193,71.815025,76.8,2401.838127,-0.903128,0.987693,5.844207,0.187500,0.268473,0.499179,85.312375,14.866221,0.350622,0.298923


In [10]:
# Best sliders by whiff rate
sliders = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'SL']
print("Top 10 sliders by whiff rate:")
sliders.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'hmov_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 sliders by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,hmov_avg,vmov_avg,whiff_pct
6381,"Natera Jr., Samy",20,0.322581,86.165000,-0.399000,0.552500,0.714286
3830,"Espada, Jose",25,0.287356,82.664000,-0.136400,-0.283600,0.700000
3798,"Javier, Cristian",21,0.002232,82.761905,0.284762,0.802857,0.666667
4302,"Little, Jack",23,0.306667,85.769565,0.313043,0.158696,0.666667
3400,"Duran, Jhoan",41,0.009419,89.292683,0.095122,0.168293,0.642857
5725,"Plymell, Chase",24,0.545455,82.483333,0.200000,0.262083,0.636364
5980,"Marrero, Andrew",24,0.774194,82.733333,0.482917,-0.171667,0.636364
5187,"Garcia, Luis",102,0.017152,85.039216,0.458725,0.036569,0.633333
5550,"Villalobos, Eli",24,0.260870,83.350000,0.373750,-0.114583,0.625000
529,"Ramírez, Erasmo",55,0.017805,81.480000,0.309455,0.391091,0.612903


In [11]:
# Best changeups by whiff rate
changeups = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'CH']
print("Top 10 changeups by whiff rate:")
changeups.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 changeups by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,vmov_avg,whiff_pct
1011,"Booser, Cam",24,0.016949,87.362500,0.419167,0.750000
3989,"Thompson, Mason",22,0.010929,87.500000,0.546818,0.750000
1545,"Scott, Tayler",20,0.008254,87.665000,-0.014500,0.666667
1761,"Rucinski, Drew",27,0.077143,83.588889,0.525185,0.666667
6507,"Garnett, Tristan",22,0.511628,81.140909,0.885000,0.642857
6455,"Granillo, Andre",29,0.062635,87.931034,0.637586,0.636364
2970,"Griffin, Foster",35,0.127273,84.154286,0.723429,0.583333
4936,"Criswell, Jeff",51,0.144068,86.778431,0.466471,0.550000
2495,"Kinley, Tyler",106,0.024357,87.357547,0.923302,0.540541
6508,"Gastelum, Luis",21,0.362069,82.152381,-0.365714,0.538462


## 4. Batter Pitch Performance Profiles

How batters perform against different pitch characteristics:
- By pitch type group (fastball, breaking, offspeed)
- By velocity bucket (90-93, 94-96, 97+)
- By movement type (high rise, heavy sweep, heavy drop)
- By pitcher handedness (vs LHP, vs RHP)

In [12]:
# Get batter pitch stats (uses cached pitches)
batter_profiles = get_batter_pitch_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batters with pitch profiles: {len(batter_profiles)}")
batter_profiles.head(10)

Computing batter pitch stats...
  Computed pitch stats for 1291 batters
Batters with pitch profiles: 1291


,batter_id,batter_name,stand,total_pitches,swing_pct,whiff_pct,contact_pct,csw_pct,zone_swing_pct,chase_pct,avg_exit_velo,max_exit_velo,avg_launch_angle,xwoba,xba,barrel_pct,vs_ff_whiff_pct,vs_ff_contact_pct,vs_ff_xwoba,vs_si_whiff_pct,vs_si_contact_pct,vs_si_xwoba,vs_fc_whiff_pct,vs_fc_contact_pct,vs_fc_xwoba,...,vs_st_xwoba,velo_90_93_pitches,velo_90_93_whiff_pct,velo_90_93_swing_pct,velo_94_96_pitches,velo_94_96_whiff_pct,velo_94_96_swing_pct,velo_97_plus_pitches,velo_97_plus_whiff_pct,velo_97_plus_swing_pct,vs_LHP_pitches,vs_LHP_whiff_pct,vs_LHP_xwoba,vs_RHP_pitches,vs_RHP_whiff_pct,vs_RHP_xwoba,vs_fs_whiff_pct,vs_fs_contact_pct,vs_fs_xwoba,vs_sv_whiff_pct,vs_sv_contact_pct,vs_sv_xwoba,vs_kn_whiff_pct,vs_kn_contact_pct,vs_kn_xwoba
0,405395,,R,2620,0.478244,0.198723,0.801277,0.269466,0.635703,0.337192,90.773828,112.9,13.892578,0.381008,0.328104,0.033138,0.115741,0.884259,0.415883,0.129353,0.870647,0.35543,0.123596,0.876404,0.350156,...,0.183397,645.0,0.109375,0.496124,403.0,0.101770,0.560794,131.0,0.222222,0.618321,1151.0,0.182143,0.459617,1469.0,0.212121,0.318509,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,408234,,R,5215,0.493576,0.238539,0.761461,0.272291,0.697802,0.298463,90.282476,112.6,7.739769,0.347429,0.328076,0.023958,0.197015,0.802985,0.388774,0.103529,0.896471,0.277016,0.309278,0.690722,0.276434,...,0.331420,1569.0,0.185829,0.476737,922.0,0.151822,0.535792,279.0,0.226519,0.648746,1471.0,0.235469,0.355101,3744.0,0.239622,0.344737,0.454545,0.545455,0.252891,NaN,NaN,NaN,NaN,NaN,NaN
2,425784,,R,286,0.503497,0.368056,0.631944,0.360140,0.656051,0.317829,84.883721,106.5,8.581395,0.392586,0.337121,0.000000,0.280000,0.720000,0.499254,0.333333,0.666667,0.2765,NaN,NaN,NaN,...,NaN,91.0,0.339623,0.582418,46.0,0.347826,0.500000,NaN,NaN,NaN,125.0,0.396825,0.461009,161.0,0.345679,0.342012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,425794,,R,287,0.567944,0.306748,0.693252,0.331010,0.701389,0.433566,63.797959,109.8,-15.551020,0.253683,0.235565,0.020408,0.188406,0.811594,0.360073,0.240000,0.760000,0.18019,NaN,NaN,NaN,...,NaN,83.0,0.173077,0.626506,48.0,0.259259,0.562500,NaN,NaN,NaN,68.0,0.315789,0.147900,219.0,0.304000,0.29776,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,425877,,R,2711,0.576909,0.213555,0.786445,0.249723,0.758136,0.396615,86.694343,108.2,15.911814,0.323888,0.307536,0.016611,0.210325,0.789675,0.368055,0.098859,0.901141,0.274684,0.310606,0.689394,0.277276,...,0.368370,677.0,0.189526,0.592319,510.0,0.203175,0.617647,128.0,0.253012,0.648438,528.0,0.205224,0.343592,2183.0,0.215278,0.319075,0.315789,0.684211,0.178000,NaN,NaN,NaN,NaN,NaN,NaN
5,429664,,L,465,0.565591,0.201521,0.798479,0.234409,0.743119,0.408907,88.617895,112.0,4.484211,0.289116,0.303088,0.010526,0.184783,0.815217,0.290449,0.027778,0.972222,0.262162,0.071429,0.928571,0.208556,...,NaN,122.0,0.135135,0.606557,77.0,0.210526,0.493506,30.0,0.125000,0.533333,NaN,NaN,NaN,423.0,0.200837,0.291319,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,435559,,R,1529,0.485284,0.192722,0.807278,0.251799,0.692206,0.282383,84.805051,105.0,17.875421,0.273133,0.256906,0.013378,0.105263,0.894737,0.266977,0.061538,0.938462,0.287879,0.108108,0.891892,0.310282,...,0.030400,402.0,0.091837,0.487562,276.0,0.103448,0.525362,75.0,0.100000,0.533333,552.0,0.194553,0.281681,977.0,0.191753,0.267803,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,443558,,R,4877,0.511995,0.293152,0.706848,0.279680,0.724481,0.320468,91.758412,117.5,9.856805,0.411106,0.343142,0.023585,0.218942,0.781058,0.455874,0.156915,0.843085,0.369275,0.248731,0.751269,0.424703,...,0.312097,1188.0,0.215488,0.500000,901.0,0.207627,0.523862,273.0,0.246575,0.534799,1801.0,0.279476,0.431768,3076.0,0.301075,0.398756,0.407407,0.592593,0.304596,NaN,NaN,NaN,NaN,NaN,NaN
8,444482,,L,6682,0.500299,0.220461,0.779539,0.263095,0.697071,0.310811,89.499218,113.8,9.453479,0.351275,0.323065,0.015601,0.155689,0.844311,0.377505,0.088670,0.911330,0.343568,0.170213,0.829787,0.383771,...,0.337606,1619.0,0.162634,0.459543,1379.0,0.143243,0.536621,355.0,0.153465,0.569014,924.0,0.230769,0.3

In [13]:
# All available batter profile columns
print("Batter profile columns:")
print(batter_profiles.columns.tolist())

Batter profile columns:
['batter_id', 'batter_name', 'stand', 'total_pitches', 'swing_pct', 'whiff_pct', 'contact_pct', 'csw_pct', 'zone_swing_pct', 'chase_pct', 'avg_exit_velo', 'max_exit_velo', 'avg_launch_angle', 'xwoba', 'xba', 'barrel_pct', 'vs_ff_whiff_pct', 'vs_ff_contact_pct', 'vs_ff_xwoba', 'vs_si_whiff_pct', 'vs_si_contact_pct', 'vs_si_xwoba', 'vs_fc_whiff_pct', 'vs_fc_contact_pct', 'vs_fc_xwoba', 'vs_sl_whiff_pct', 'vs_sl_contact_pct', 'vs_sl_xwoba', 'vs_cu_whiff_pct', 'vs_cu_contact_pct', 'vs_cu_xwoba', 'vs_ch_whiff_pct', 'vs_ch_contact_pct', 'vs_ch_xwoba', 'vs_kc_whiff_pct', 'vs_kc_contact_pct', 'vs_kc_xwoba', 'vs_st_whiff_pct', 'vs_st_contact_pct', 'vs_st_xwoba', 'velo_90_93_pitches', 'velo_90_93_whiff_pct', 'velo_90_93_swing_pct', 'velo_94_96_pitches', 'velo_94_96_whiff_pct', 'velo_94_96_swing_pct', 'velo_97_plus_pitches', 'velo_97_plus_whiff_pct', 'velo_97_plus_swing_pct', 'vs_LHP_pitches', 'vs_LHP_whiff_pct', 'vs_LHP_xwoba', 'vs_RHP_pitches', 'vs_RHP_whiff_pct', 'vs_RH

In [14]:
# Summary of key batter metrics
batter_stats = [
    'whiff_pct', 'contact_pct', 'chase_pct', 'zone_swing_pct',
    'fastball_whiff_pct', 'breaking_whiff_pct', 'offspeed_whiff_pct',
    'velo_97_plus_whiff_pct', 'high_rise_whiff_pct', 'heavy_sweep_whiff_pct',
    'avg_exit_velo', 'xwoba',
]

available_stats = [c for c in batter_stats if c in batter_profiles.columns]
batter_profiles[available_stats].describe().round(3)

,whiff_pct,contact_pct,chase_pct,zone_swing_pct,velo_97_plus_whiff_pct,avg_exit_velo,xwoba
count,1291.000,1291.000,1291.000,1291.000,839.000,1079.000,1079.000
mean,0.260,0.740,0.292,0.670,0.221,86.733,0.350
std,0.076,0.076,0.071,0.070,0.084,4.592,0.060
min,0.063,0.392,0.070,0.351,0.000,55.809,0.128
25%,0.211,0.697,0.245,0.632,0.163,85.356,0.312
50%,0.253,0.747,0.289,0.673,0.215,87.537,0.348
75%,0.303,0.789,0.331,0.716,0.270,89.339,0.388
max,0.608,0.937,0.681,0.900,0.562,95.963,0.623


In [15]:
# Best contact rates (lowest whiff)
print("Top 10 contact rates (lowest whiff%):")
batter_profiles.nsmallest(10, 'whiff_pct')[[
    'batter_id', 'stand', 'total_pitches',
    'whiff_pct', 'contact_pct', 'chase_pct', 'avg_exit_velo'
]]

Top 10 contact rates (lowest whiff%):


,batter_id,stand,total_pitches,whiff_pct,contact_pct,chase_pct,avg_exit_velo
550,650333,L,11941,0.063486,0.936514,0.297756,86.945622
214,591971,L,411,0.065359,0.934641,0.179104,84.010390
306,605200,R,153,0.066667,0.933333,0.115385,65.558333
1037,680757,L,11175,0.080725,0.919275,0.207019,84.903060
656,663611,R,3093,0.085106,0.914894,0.294194,84.802029
11,445055,L,54,0.085714,0.914286,0.518519,NaN
1263,800325,R,87,0.093023,0.906977,0.238095,NaN
1277,805779,R,2209,0.094033,0.905967,0.332726,84.797683
691,664058,R,3335,0.096263,0.903737,0.303915,81.300231
741,665877,R,977,0.096447,0.903553,0.229614,84.552551


In [16]:
# Batters who struggle vs 97+ velocity
if 'velo_97_plus_whiff_pct' in batter_profiles.columns:
    has_velo_data = batter_profiles['velo_97_plus_whiff_pct'].notna()
    print("Batters who struggle most vs 97+ mph:")
    batter_profiles[has_velo_data].nlargest(10, 'velo_97_plus_whiff_pct')[[
        'batter_id', 'stand', 'velo_97_plus_pitches',
        'velo_97_plus_whiff_pct', 'whiff_pct'
    ]]

Batters who struggle most vs 97+ mph:


In [17]:
# Batters who struggle vs breaking balls
if 'breaking_whiff_pct' in batter_profiles.columns:
    has_brk_data = batter_profiles['breaking_whiff_pct'].notna()
    print("Batters who struggle most vs breaking balls:")
    batter_profiles[has_brk_data].nlargest(10, 'breaking_whiff_pct')[[
        'batter_id', 'stand', 'breaking_pitches',
        'breaking_whiff_pct', 'whiff_pct'
    ]]

## 5. Batter Stats by Pitch Type

Detailed breakdown for each pitch type a batter faces.

In [ ]:
# Get per-pitch-type stats for batters
batter_pitch_types = get_batter_pitch_type_stats(
    DATA_START, DATA_END,
    min_pitches=50,  # Higher threshold for multi-season data
    pitches_df=pitches
)

print(f"Batter-pitch type combinations: {len(batter_pitch_types)}")
batter_pitch_types.head(15)

Computing batter pitch-type stats...


In [ ]:
# Best fastball hitters (lowest whiff on FF)
ff_stats = batter_pitch_types[batter_pitch_types['pitch_type'] == 'FF']
if len(ff_stats) > 0 and 'xwoba' in ff_stats.columns:
    ff_qualified = ff_stats[ff_stats['pitch_count'] >= 100]
    print("Best fastball hitters by xwOBA:")
    ff_qualified.nlargest(10, 'xwoba')[[
        'batter_id', 'pitch_count', 'whiff_pct', 'contact_pct',
        'avg_exit_velo', 'xwoba'
    ]]

Best fastball hitters by xwOBA:


## 6. Plate Appearance Outcomes

Extract completed plate appearances with outcomes (K, BB, 1B, 2B, 3B, HR, OUT).

In [ ]:
# Get plate appearances with outcomes
plate_apps = get_plate_appearances(
    DATA_START, DATA_END,
    pitches_df=pitches
)

print(f"Total plate appearances: {len(plate_apps):,}")
plate_apps.head(15)

In [ ]:
# Outcome distribution
print("PA outcome distribution:")
outcome_counts = plate_apps['outcome'].value_counts()
outcome_pcts = plate_apps['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

PA outcome distribution:


,count,pct
outcome,,
OUT,2301,47.0
K,1115,22.8
1B,663,13.5
BB,406,8.3
2B,199,4.1
HR,120,2.5
HBP,48,1.0
3B,25,0.5
OTHER,18,0.4


In [ ]:
# Unique matchup counts
print(f"Unique pitchers: {plate_apps['pitcher_id'].nunique():,}")
print(f"Unique batters: {plate_apps['batter_id'].nunique():,}")

# Pitcher-batter combinations
matchups = plate_apps.groupby(['pitcher_id', 'batter_id']).size()
print(f"Unique pitcher-batter matchups: {len(matchups):,}")
print(f"\nPAs per matchup distribution:")
matchups.describe()

Unique pitchers: 382
Unique batters: 394
Unique pitcher-batter matchups: 2,976

PAs per matchup distribution:


count    2976.000000
mean        1.644825
std         0.804521
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
dtype: float64

## 7. Save Data

Save the computed profiles for use in model training.

In [ ]:
from pathlib import Path

# Create output directory
output_dir = Path(f'../{DATA_DIR}/profiles')
output_dir.mkdir(parents=True, exist_ok=True)

# Save profiles
pitcher_arsenal.to_csv(output_dir / 'pitcher_arsenal.csv', index=False)
pitcher_pitch_types.to_csv(output_dir / 'pitcher_pitch_types.csv', index=False)
batter_profiles.to_csv(output_dir / 'batter_profiles.csv', index=False)
batter_pitch_types.to_csv(output_dir / 'batter_pitch_types.csv', index=False)
plate_apps.to_parquet(output_dir / 'plate_appearances.parquet', index=False)

print(f"Saved data to {output_dir.resolve()}:")
print(f"  - pitcher_arsenal.csv: {len(pitcher_arsenal):,} rows")
print(f"  - pitcher_pitch_types.csv: {len(pitcher_pitch_types):,} rows")
print(f"  - batter_profiles.csv: {len(batter_profiles):,} rows")
print(f"  - batter_pitch_types.csv: {len(batter_pitch_types):,} rows")
print(f"  - plate_appearances.parquet: {len(plate_apps):,} rows")